In [ ]:
import pandas as pd
import requests
import csv
from datetime import datetime, timedelta
from tqdm import tqdm
import os

In [ ]:
url_table = 'https://earthquake.usgs.gov/fdsnws/event/1/query'
url_count = "https://earthquake.usgs.gov/fdsnws/event/1/count"
start_date = '2025-12-01'
end_date = '2025-12-31'
format = "csv"
folder = "tmp"
start_date_range, end_time_range = "", ""
count_all = 0
df_res = pd.DataFrame(columns=['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst',
       'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type',
       'horizontalError', 'depthError', 'magError', 'magNst', 'status',
       'locationSource', 'magSource'])

In [ ]:
os.makedirs(folder, exist_ok=True)
for dirpath, dirnames, filenames in os.walk(folder, topdown=False):
    for filename in filenames:
        file_to_remove = os.path.join(dirpath, filename)
        os.remove(file_to_remove)

In [ ]:
delta = datetime.strptime(end_date, "%Y-%m-%d") - datetime.strptime(start_date, "%Y-%m-%d")

In [ ]:
for i in tqdm(range(1, delta.days + 1)):
    start_date_range, end_time_range =(datetime.strptime(start_date, "%Y-%m-%d") + timedelta(days = i - 1)).strftime("%Y-%m-%d"), (datetime.strptime(start_date, "%Y-%m-%d") + timedelta(days = i)).strftime("%Y-%m-%d")
    response_table = requests.get(url=url_table, params={"starttime": start_date_range, "endtime" : end_time_range, "format": format})
    response_count = requests.get(url=url_count, params={"starttime": start_date_range, "endtime" : end_time_range, "format": format})
    list_res_table = response_table.content.decode('utf-8').splitlines()
    count_res = response_count.content.decode("utf-8")
    list_res_sep = list(csv.reader(list_res_table, delimiter=','))
    df = pd.DataFrame(columns=df_res.columns, data=list_res_sep[1:])
    if response_table.status_code == 200 and response_count.status_code == 200:
       if len(df) == int(count_res):   
        df_res = pd.concat([df_res, df]) 
        count_all += int(count_res)
    else:
        print(response_table.status_code, response_count.status_code)
    
df_res['update_at'] =  pd.to_datetime(pd.Timestamp.now()).date()
df_res['date_report'] = pd.to_datetime(df_res['time']).dt.date
if count_all == len(df_res):
   os.makedirs(folder, exist_ok=True)
   df_res.to_csv(f'{folder}/csv_MadMaxD_{start_date}_{end_date}_{pd.Timestamp.now().strftime("%Y-%m-%d_%H_%M")}.csv', sep=';', index=False)
